# Lab 03: K-Nearest Neighbors - Classification

This lab continues directly from Lab 02. You will turn distance computations
into a complete k-NN classifier, study how the choice of `k` changes decision
boundaries, and evaluate `k=1` and `k=5` on CIFAR-10.

Before starting, copy your completed implementations of
`compute_distances_two_loops`, `compute_distances_one_loop`, and
`compute_distances_no_loops` from the Lab 02 `knn.py` into this lab's
`knn.py`. Do not copy output cells or solutions from another source.


## Setup

Run this cell from the `A1` directory. It imports the local support package and
checks that the lab file is visible.


In [ ]:
import torch
import matplotlib.pyplot as plt
import dlcv2026

plt.rcParams["figure.figsize"] = (10.0, 8.0)
plt.rcParams["font.size"] = 16

from knn import hello
hello()


## Confirm your Lab 02 distance function

This independent check must pass before you implement label prediction. If it
fails, revisit the distance functions you copied from Lab 02.


In [ ]:
from knn import compute_distances_no_loops

x_train_small = torch.tensor([[0.0, 0.0], [3.0, 4.0]], dtype=torch.float64)
x_test_small = torch.tensor([[0.0, 4.0], [3.0, 0.0]], dtype=torch.float64)
expected = torch.tensor([[16.0, 9.0], [9.0, 16.0]], dtype=torch.float64)
actual = compute_distances_no_loops(x_train_small, x_test_small)
assert torch.allclose(actual, expected), f"Expected {expected}, got {actual}"
print("Distance function ready for Lab 03.")


## Predict labels
Now that we have a method for computing distances between training and test examples, we need to implement a function that uses those distances together with the training labels to predict labels for test samples.

In the file `knn.py`, implement the function `predict_labels`.

In [ ]:
from knn import predict_labels

torch.manual_seed(0)
dists = torch.tensor([
    [0.3, 0.4, 0.1],
    [0.1, 0.5, 0.5],
    [0.4, 0.1, 0.2],
    [0.2, 0.2, 0.4],
    [0.5, 0.3, 0.3],
])
y_train = torch.tensor([0, 1, 0, 1, 2])
y_pred_expected = torch.tensor([1, 0, 0])
y_pred = predict_labels(dists, y_train, k=3)
correct = y_pred.tolist() == y_pred_expected.tolist()
print('Correct: ', correct)

Now we have implemented all the required functionality for the K-Nearest Neighbor classifier. In the file `knn.py`, complete the implementation of the `KnnClassifier` class.

We can get some intuition into the KNN classifier by visualizing its predictions on toy 2D data. Here we will generate some random training and test points in 2D, and assign random labels to the training points. We can then make predictions for the test points, and visualize both training and test points. Training points are shown as stars, and test points are shown as small transparent circles. The color of each point denotes its label -- ground-truth label for training points, and predicted label for test points.

In [ ]:
from knn import KnnClassifier

num_test = 10000
num_train = 20
num_classes = 5

# Generate random training and test data
torch.manual_seed(128)
x_train = torch.rand(num_train, 2)
y_train = torch.randint(num_classes, size=(num_train,))
x_test = torch.rand(num_test, 2)
classifier = KnnClassifier(x_train, y_train)

# Plot predictions for different values of k
for k in [1, 3, 5]:
    y_test = classifier.predict(x_test, k=k)
    plt.gcf().set_size_inches(8, 8)
    class_colors = ['r', 'g', 'b', 'k', 'y']
    train_colors = [class_colors[c] for c in y_train]
    test_colors = [class_colors[c] for c in y_test]
    plt.scatter(x_test[:, 0], x_test[:, 1],
                color=test_colors, marker='o', s=32, alpha=0.05)
    plt.scatter(x_train[:, 0], x_train[:, 1],
                color=train_colors, marker='*', s=128.0)
    plt.title('Predictions for k = %d' % k, size=16)
    plt.show()

We can use the exact same KNN code to perform image classification on CIFAR-10!

Now let's put everything together and test our k-NN classifier on a subset of CIFAR-10, using k=1:

If you've implemented everything correctly you should see an accuracy of about 27%.

In [ ]:
from knn import KnnClassifier

torch.manual_seed(0)
num_train = 5000
num_test = 500
x_train, y_train, x_test, y_test = dlcv2026.data.cifar10(num_train, num_test)

classifier = KnnClassifier(x_train, y_train)
classifier.check_accuracy(x_test, y_test, k=1)

Now let's increase to k=5. You should see a slightly higher accuracy than k=1:

In [ ]:
from knn import KnnClassifier

torch.manual_seed(0)
num_train = 5000
num_test = 500
x_train, y_train, x_test, y_test = dlcv2026.data.cifar10(num_train, num_test)

classifier = KnnClassifier(x_train, y_train)
classifier.check_accuracy(x_test, y_test, k=5)

## Wrap-up

Record short answers before leaving:

1. How did the decision boundary change as `k` increased from 1 to 5?
2. Which value performed better on the CIFAR-10 subset?
3. Why is test-set accuracy not the right quantity for choosing `k`?

The homework continuation will use validation folds to choose `k` without
tuning on the test set.
